# Demo: Demand Modeling

This notebook demonstrates `calculate_treatment_scores` and `create_demand_table` with a handcrafted symptom mapping.

`generate_symptom_mapping` is optional here because it requires a live OpenAI API key.

In [ ]:
from pathlib import Path
import sys

def add_app_python_src_to_path():
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / "dais-health-access" / "python" / "src"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            return candidate
    raise FileNotFoundError("Could not find dais-health-access/python/src from the current working directory")

APP_SRC = add_app_python_src_to_path()

In [ ]:
import os
import pandas as pd
from facility_prioritization.config import load_config
from facility_prioritization.demand_modeling import calculate_treatment_scores, create_demand_table, generate_symptom_mapping

config = load_config(APP_SRC.parent.parent / "config" / "config.yaml")

cleaned_survey = pd.DataFrame(
    [
        {"district_name": "Alpha", "state_ut": "State 1", "respiratory_need": 10.5, "cancer_screening_gap": 2.0, "childhood_care_gap": 8.0},
        {"district_name": "Beta", "state_ut": "State 1", "respiratory_need": 7.1, "cancer_screening_gap": 9.0, "childhood_care_gap": 3.0},
        {"district_name": "Gamma", "state_ut": "State 2", "respiratory_need": 12.2, "cancer_screening_gap": 4.5, "childhood_care_gap": 6.0},
    ]
)

symptom_mapping = pd.DataFrame(
    {
        "respiratory_need": [1, 0, 0],
        "cancer_screening_gap": [0, 1, 0],
        "childhood_care_gap": [0, 0, 1],
        "reasoning": ["Respiratory burden", "Cancer burden", "Child health burden"],
    },
    index=["cardiology", "oncology", "pediatrics"],
)

scores_dict, score_warnings = calculate_treatment_scores(cleaned_survey, symptom_mapping, config=config)
demand_table, demand_warnings = create_demand_table(scores_dict, config=config)

print("Warnings:")
for warning in score_warnings + demand_warnings:
    print(f"- {warning}")

demand_table.head(9)

In [ ]:
# Optional: live mapping generation if you have OPENAI_API_KEY set.
if os.getenv("OPENAI_API_KEY"):
    live_mapping, live_warnings = generate_symptom_mapping(
        treatment_list=["cardiology", "oncology"],
        survey_columns=cleaned_survey.columns.tolist(),
        config=config,
    )
    print(live_warnings)
    live_mapping
else:
    print("Skipping generate_symptom_mapping demo because OPENAI_API_KEY is not set.")